# Task 1a: Perform Text Generation

In this notebook, you learn how to use Large Language Model (LLM) to generate an email response to a customer who provided negative feedback on the quality of customer service they received from the support engineer. In this notebook, you generate an email with a thank you note based on the customer's previous email. You use the Amazon Titan model using the Amazon Bedrock API with Boto3 client.

The prompt used in this task is called a zero-shot prompt. In a zero-shot prompt, you describe the task or desired output to the language model in plain language. The model then uses its pre-trained knowledge and capabilities to generate a response or complete the task based solely on the provided prompt.

#### Scenario
You are Bob a Customer Service Manager at AnyCompany and some of your customers are not happy with the customer service and are providing negative feedbacks on the service provided by customer support engineers. Now, you would like to respond to those customers apologizing for the poor service and to regain trust. You need the help of an LLM to generate a bulk of emails for you which are human friendly and personalized to the customer's sentiment from previous email correspondence.

## Task 1a.1: Environment setup

In this task, you set up your environment.

In [1]:
#Create a service client by name using the default session.
import json
import os
import sys

import boto3
import botocore

module_path = ".."
sys.path.append(os.path.abspath(module_path))

bedrock_client = boto3.client('bedrock-runtime',region_name=os.environ.get("AWS_DEFAULT_REGION", None))

## Task 1a.2: Generate text

In this task, you prepare an input for the Amazon Bedrock service to generate an email.

In [33]:
# create the prompt
prompt_data = """
Command: Write an email from Bob, Customer Service Manager, AnyCompany to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer"""

In [34]:
body = json.dumps({
    "inputText": prompt_data, 
    "textGenerationConfig":{
        "maxTokenCount":8192,
        "stopSequences":[],
        "temperature":0,
        "topP":0.9
        }
    }) 

Next, you use the Amazon Titan model.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **Note:** Amazon Titan supports a context window of ~4k tokens and accepts the following parameters:
- `inputText`: Prompt to the LLM
- `textGenerationConfig`: These are the parameters that model will take into account while generating the output.

The Amazon Bedrock API provides you with an API `invoke_model` which accepts the following:
- `modelId`: This is the model ARN for the various foundation models available under Amazon Bedrock
- `accept`: The type of input request
- `contentType`: The content type of the output
- `body`: A json string consisting of the prompt and the configurations

Refer to [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids-arns.html) for Available text generation model Ids.

## Task 1a.3: Invoke the Amazon Titan Large language model

In this task, you explore how the model generates an output based on the prompt created earlier.

### Complete Output Generation

This email is generated using the Amazon Titan model by understanding the input request and utilizing its inherent understanding of different modalities. The request to the API is synchronous and waits for the entire output to be generated by the model.

In [57]:

prompt_data = """

Write an email from Bob, Customer Service Manager, AnyCompany to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer
"""



In [58]:
body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt_data
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 8192,
        "temperature": 0.3,
        "topP": 0.9
    }
})


In [59]:
import boto3
import json
import botocore

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-2"
)

modelId = "amazon.nova-lite-v1:0"

accept = "application/json"
contentType = "application/json"

outputText = ""

try:

    response = bedrock_client.invoke_model(
        body=body,
        modelId=modelId,
        accept=accept,
        contentType=contentType
    )

    response_body = json.loads(
        response["body"].read()
    )

    outputText = response_body["output"]["message"]["content"][0]["text"]

    print(outputText)

except botocore.exceptions.ClientError as error:

    print("AWS Error:", error.response["Error"]["Code"])
    print("Message:", error.response["Error"]["Message"])


Subject: Addressing Your Recent Experience with Our Customer Support

Dear John Doe,

I hope this email finds you well.

I am writing to you in response to the feedback you recently provided regarding your experience with our customer support team. First and foremost, I want to express our sincere apologies for any inconvenience or frustration you may have encountered. At AnyCompany, we strive to deliver exceptional service, and it is clear that we fell short in your recent interaction.

We take your feedback very seriously and are committed to making things right. I have reviewed the details of your case and would like to personally address the issues you encountered. Our goal is to ensure that you have a positive experience moving forward.

Here are the steps we will be taking to resolve the issues:

1. **Immediate Review:** We have initiated a thorough review of the service provided by our customer support engineer to identify where we went wrong and how we can improve.
   
2. **Dir

In [60]:
# The relevant portion of the response begins after the first newline character
# Below we print the response beginning after the first occurence of '\n'.

email = outputText[outputText.index('\n')+1:]
print(email)



Dear John Doe,

I hope this email finds you well.

I am writing to you in response to the feedback you recently provided regarding your experience with our customer support team. First and foremost, I want to express our sincere apologies for any inconvenience or frustration you may have encountered. At AnyCompany, we strive to deliver exceptional service, and it is clear that we fell short in your recent interaction.

We take your feedback very seriously and are committed to making things right. I have reviewed the details of your case and would like to personally address the issues you encountered. Our goal is to ensure that you have a positive experience moving forward.

Here are the steps we will be taking to resolve the issues:

1. **Immediate Review:** We have initiated a thorough review of the service provided by our customer support engineer to identify where we went wrong and how we can improve.
   
2. **Direct Contact:** I will be in direct contact with you over the next few

### Streaming Output Generation

Bedrock also supports that the output can be streamed as it is generated by the model in form of chunks. This email is generated by invoking the model with streaming option. `invoke_model_with_response_stream` returns a `ResponseStream` which you can read from.

In [61]:
# Invoke model with response stream

output = []

try:
    response = bedrock_client.invoke_model_with_response_stream(
        body=body,
        modelId=modelId,
        accept=accept,
        contentType=contentType
    )

    stream = response.get("body")

    i = 1

    if stream:
        for event in stream:

            chunk = event.get("chunk")

            if chunk:
                chunk_obj = json.loads(chunk.get("bytes").decode("utf-8"))

                # Nova Lite streaming response format
                content_delta = chunk_obj.get("contentBlockDelta")

                if content_delta:
                    text = content_delta.get("delta", {}).get("text", "")

                    if text:
                        output.append(text)

                        print(
                            f"\t\t\x1b[31m**Chunk {i}**\x1b[0m\n{text}\n"
                        )
                        i += 1

except botocore.exceptions.ClientError as error:

    if error.response["Error"]["Code"] == "AccessDeniedException":

        print(
            f"\x1b[41m{error.response['Error']['Message']}\x1b[0m\n"
            "To troubleshoot this issue, please refer to:\n"
            "https://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\n"
            "https://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html"
        )

    else:
        raise error

		**Chunk 1**
Subject

		**Chunk 2**
: Addressing

		**Chunk 3**
 Your

		**Chunk 4**
 Recent

		**Chunk 5**
 Feedback

		**Chunk 6**
 and Our Commitment

		**Chunk 7**
 to Improvement

		**Chunk 8**


Dear John Doe,

I

		**Chunk 9**
 hope this email finds you well

		**Chunk 10**
.

I am writing to you

		**Chunk 11**
 in response to the

		**Chunk 12**
 feedback you recently

		**Chunk 13**
 provided regarding

		**Chunk 14**
 the

		**Chunk 15**
 service you

		**Chunk 16**
 received from our customer support

		**Chunk 17**
 engineer

		**Chunk 18**
. First

		**Chunk 19**
 and foremost, I

		**Chunk 20**
 want to express

		**Chunk 21**
 our

		**Chunk 22**
 sincere apologies for any

		**Chunk 23**
 inconvenience

		**Chunk 24**
 or frustration you experienced.

		**Chunk 25**
 At

		**Chunk 26**
 Any

		**Chunk 27**
Company, we strive to deliver

		**Chunk 28**
 exceptional service,

		**Chunk 29**
 and it

		**Chunk 30**
 is clear that we fell

		**Chunk 31**
 short of

		**Ch

In [62]:
# Invoke model with response stream

import json
import botocore

# Prompt related to the John Doe customer-service scenario
prompt_data = """
Write an email from Bob, Customer Service Manager, AnyCompany to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer"""

# Make sure the request body uses Nova format
body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt_data
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.5,
        "topP": 0.9
    }
})

output = []

try:

    response = bedrock_client.invoke_model_with_response_stream(
        body=body,
        modelId=modelId,
        accept=accept,
        contentType=contentType
    )

    stream = response.get("body")

    i = 1

    if stream:
        for event in stream:

            chunk = event.get("chunk")

            if chunk:
                chunk_obj = json.loads(
                    chunk.get("bytes").decode()
                )

                # Extract Nova streaming response
                content_delta = chunk_obj.get("contentBlockDelta")

                if content_delta:
                    text = content_delta.get("delta", {}).get("text")

                    if text:
                        output.append(text)

                        print(
                            f"Chunk {i}: {text}",
                            end="",
                            flush=True
                        )

                        i += 1

    print("\n\nStreaming completed!")

    # Combine all generated chunks into one complete email
    complete_response = "".join(output)

except botocore.exceptions.ClientError as error:

    print("AWS Error:", error.response["Error"]["Code"])
    print("Message:", error.response["Error"]["Message"])

    if error.response["Error"]["Code"] == "AccessDeniedException":
        print("Check your IAM permissions and Bedrock model access.")


Chunk 1: SubjectChunk 2: : AddressingChunk 3:  YourChunk 4:  RecentChunk 5:  FeedbackChunk 6: 

Dear John DoeChunk 7: ,

I hope this messageChunk 8:  finds you well.

I amChunk 9:  writing to youChunk 10:  inChunk 11:  response to theChunk 12:  feedbackChunk 13:  you recentlyChunk 14:  provided regardingChunk 15:  theChunk 16:  service youChunk 17:  received from our customerChunk 18:  support engineerChunk 19: . FirstChunk 20:  andChunk 21:  foremost, I wantChunk 22:  to sincerely apologizeChunk 23:  for any inconvenienceChunk 24:  or frustration thisChunk 25:  experienceChunk 26:  mayChunk 27:  have caused you. AtChunk 28:  AnyChunk 29: Company, we strive to provideChunk 30:  theChunk 31:  highest level of serviceChunk 32:  to our customers,Chunk 33:  and itChunk 34:  is clear that we fellChunk 35:  short inChunk 36:  yourChunk 37:  recentChunk 38:  interactionChunk 39: .

YourChunk 40:  feedback is incrediblyChunk 41:  important to us,Chunk 42:  asChunk 43:  it helpsChunk 44:  us id

The stream with response approach helps to quickly obtain the output of the model and allows the service to complete it as you read. This assists in use cases where you request the model to generate longer pieces of text. You can later combine all the chunks generated to form the complete output and use it for your use case. 

In [63]:
#combine output chunks
print('\t\t\x1b[31m**COMPLETE OUTPUT**\x1b[0m\n')
complete_output = ''.join(output)
print(complete_output)

		**COMPLETE OUTPUT**

Subject: Addressing Your Recent Feedback

Dear John Doe,

I hope this message finds you well.

I am writing to you in response to the feedback you recently provided regarding the service you received from our customer support engineer. First and foremost, I want to sincerely apologize for any inconvenience or frustration this experience may have caused you. At AnyCompany, we strive to provide the highest level of service to our customers, and it is clear that we fell short in your recent interaction.

Your feedback is incredibly important to us, as it helps us identify areas where we can improve and ensure that all our customers have a positive experience. I have taken the time to review the details of your case and would like to personally address the concerns you raised.

To resolve the issues you encountered, I have scheduled a follow-up with our customer support team to discuss the specific points you mentioned. Additionally, I will be personally overseeing t


You have now experimented with using the boto3 SDK, which provides basic exposure to the Amazon Bedrock API. Using this API, you have seen the use case of generating an email to respond to a customer's negative feedback.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

### Cleanup

You have completed this notebook. To move to the next part of the lab, do the following:

- Close this notebook file and continue with **Task1b.ipynb**.